# Модули и импорт: код в `.py`, запуск из командной строки

**Пара КТП 6** (2 ч). До сих пор функции жили в ячейках ноутбука. Сегодня они переезжают в **файл-модуль**, который можно импортировать, запускать из командной строки и проверять отдельным файлом тестов.

Минимум сдачи: `hello.py`, `metrics.py`, `main.py`, `predict.py`, `manual_tests.py` — все запускаются командой `python имя.py`.

**Как работать в этом ноутбуке**

| Ячейка начинается с | Что это |
|---|---|
| `%%writefile имя.py` | сохранить текст ячейки в файл `имя.py` (сама ячейка не выполняется как код) |
| `!python имя.py` | команда **терминала**: запустить файл заново, в отдельном процессе |
| `!ls` | команда терминала: показать файлы в папке |

Если у вас Python и терминал на компьютере — делайте то же в одной папке: файл в редакторе, команды в терминале **без** `!` (в Windows иногда `py` вместо `python`).

In [ ]:
# Те же данные, что на паре 5
PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS =      [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]


## 1. Файл .py и ноутбук .ipynb

Ноутбук `.ipynb` — это **JSON**: список ячеек с кодом, текстом и сохранёнными выводами. Примерно так выглядит одна ячейка внутри файла:

```json
{"cell_type": "code", "source": ["print(1 + 1)"], "outputs": [{"text": ["2\n"]}]}
```

Файл `.py` — **только код**, обычный текст. Python читает его сверху вниз и выполняет. Такой файл называют **программой** (если его запускают) или **модулем** (если его импортируют).

Создайте файл и запустите его.

In [ ]:
%%writefile hello.py
print("hello from file")


In [ ]:
!python hello.py

In [ ]:
# какие файлы появились в папке?
!ls

**Задание.** Поменяйте текст в `hello.py`, запустите ячейку с `%%writefile` ещё раз, потом — `!python hello.py`. Что будет, если запустить только `!python hello.py`, не пересохранив файл?

## 2. Модуль metrics.py

**Модуль** — файл с функциями. Имя модуля = имя файла без `.py`.

Перенесите в файл ниже **свои** `my_accuracy` и `confusion_counts` с пары 5 (вместо `pass`).

In [ ]:
%%writefile metrics.py
"""metrics — метрики классификации на списках 0/1 (пара 5)."""


def my_accuracy(preds, labels):
    """Доля верных предсказаний или None при разной длине."""
    pass


def confusion_counts(preds, labels):
    """(tp, fp, fn, tn) для меток 0/1."""
    pass


In [ ]:
import metrics

acc = metrics.my_accuracy(PREDICTIONS, LABELS)
print(acc)
assert abs(acc - 0.8) < 1e-9


In [ ]:
from metrics import confusion_counts

tp, fp, fn, tn = confusion_counts(PREDICTIONS, LABELS)
print(tp, fp, fn, tn)
assert (tp, fp, fn, tn) == (5, 1, 1, 3)


**Ловушка.** Модуль загружается в память **один раз** на процесс. Если вы исправили `metrics.py`, а `import metrics` в ноутбуке по-прежнему возвращает старое — перезагрузите модуль ячейкой ниже. Команда `!python main.py` этой проблемы не имеет: каждый запуск — новый процесс, файл читается заново.

In [ ]:
import importlib

importlib.reload(metrics)
print(metrics.my_accuracy(PREDICTIONS, LABELS))

## 3. Программа main.py

Программа импортирует модуль и печатает отчёт. Данные пока внутри программы.

In [ ]:
%%writefile main.py
"""Отчёт по метрикам. Запуск: python main.py"""

from metrics import my_accuracy, confusion_counts

PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

acc = my_accuracy(PREDICTIONS, LABELS)
tp, fp, fn, tn = confusion_counts(PREDICTIONS, LABELS)
print("accuracy:", acc)
print("tp fp fn tn:", tp, fp, fn, tn)


In [ ]:
!python main.py

### Эксперимент: код верхнего уровня

Допишите в **конец** `metrics.py` строку

```python
print("self-check:", my_accuracy([1, 0], [1, 1]))
```

и снова выполните `!python main.py`. Откуда взялась лишняя строка?

Всё, что в файле **не внутри функций**, выполняется при импорте. Чтобы проверка работала только при прямом запуске `python metrics.py`, её прячут под условие `if __name__ == "__main__":`. Переменная `__name__` равна `"__main__"`, когда файл запустили как программу, и равна имени модуля (`"metrics"`), когда его импортировали.

Перепишите `metrics.py`: ваши функции + self-check под условием.

In [ ]:
%%writefile metrics.py
"""metrics — метрики классификации на списках 0/1 (пара 5)."""


def my_accuracy(preds, labels):
    """Доля верных предсказаний или None при разной длине."""
    pass


def confusion_counts(preds, labels):
    """(tp, fp, fn, tn) для меток 0/1."""
    pass


if __name__ == "__main__":
    # выполняется только при запуске: python metrics.py
    print("self-check:", my_accuracy([1, 0], [1, 1]))


In [ ]:
# прямой запуск: self-check печатается
!python metrics.py

In [ ]:
# импорт из программы: self-check молчит
!python main.py

## 4. Аргументы командной строки

`sys.argv` — список **строк** из командной строки. `sys.argv[0]` — имя файла, дальше — аргументы.

Напишите `predict.py`: `python predict.py 72 60` печатает `1` (72 ≥ 60), `python predict.py 44 60` — `0`.

In [ ]:
%%writefile predict.py
"""Запуск: python predict.py БАЛЛ ПОРОГ  → печатает 1 (сдал) или 0"""

import sys

print("argv:", sys.argv)  # посмотрите, что здесь, и удалите строку


def predict_pass(score, threshold):
    pass


score = None      # int(sys.argv[1])
threshold = None  # int(sys.argv[2])
print(predict_pass(score, threshold))


In [ ]:
!python predict.py 72 60

In [ ]:
!python predict.py 44 60

**Задание.** Запустите `!python predict.py` без аргументов. Какая ошибка? Сделайте порог необязательным: если его нет — использовать `60`; если нет и балла — напечатать подсказку «использование: …».

In [ ]:
!python predict.py
!python predict.py 72

## 5. Файл тестов

Тесты — отдельный файл, который **импортирует** модуль и проверяет его через `assert`. Последняя строка печатается, только если все проверки прошли.

In [ ]:
%%writefile manual_tests.py
"""Тесты модуля metrics. Запуск: python manual_tests.py"""

from metrics import my_accuracy, confusion_counts

PREDICTIONS = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]
LABELS = [1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

assert abs(my_accuracy(PREDICTIONS, LABELS) - 0.8) < 1e-9, "accuracy на 10 объектах"
assert my_accuracy([1], [1, 0]) is None, "разная длина → None"
assert confusion_counts(PREDICTIONS, LABELS) == (5, 1, 1, 3), "tp fp fn tn"
assert sum(confusion_counts(PREDICTIONS, LABELS)) == len(PREDICTIONS), "сумма = число объектов"
print("All tests passed")


In [ ]:
!python manual_tests.py

### Сломайте и прочитайте traceback

В `metrics.py` поменяйте местами `fp` и `fn` в `return` (пересохраните файл) и запустите тесты ещё раз.

Traceback читают **снизу вверх**: последняя строка — тип ошибки и сообщение; выше — `File "...", line N` — файл и строка, где упала проверка.

Запишите в ячейку ниже: какой файл, какая строка, какое сообщение. Потом верните `metrics.py` в порядок.

**Ответ:** файл … , строка … , сообщение … 

In [ ]:
# после исправления снова должно быть All tests passed
!python manual_tests.py

## 6. Данные — тоже модуль

`data/module_datasets.py` из репозитория курса — обычный модуль с константами. Скачаем файл рядом и импортируем из него данные.

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
    "modules/08_01_functions_recursion/data/module_datasets.py",
    "module_datasets.py",
)

from module_datasets import PREDICTIONS as P_DATA, LABELS as L_DATA

print("объектов в датасете модуля:", len(P_DATA), len(L_DATA))


In [ ]:
!ls

### Мост к паре 8

Стартовый код артефакта устроен точно так же, как ваша папка сейчас:

```text
text_stats_starter/
  text_stats.py        ← модуль с функциями (как metrics.py)
  manual_tests.py      ← тесты, запуск: python manual_tests.py
  data/
    module_datasets.py ← данные-модуль
```

Домашнее задание — [homework.ipynb](homework.ipynb): та же структура на функциях пары 3.

## Итог

| Понятие | Что запомнить |
|---|---|
| `.py` vs `.ipynb` | код-текст vs JSON с ячейками и выводами |
| модуль | файл с функциями; `import metrics`, `from metrics import f` |
| `python file.py` | новый процесс, файл читается заново |
| `if __name__ == "__main__":` | код только при прямом запуске |
| `sys.argv` | аргументы — список строк |
| `manual_tests.py` | отдельный файл с `assert` |